# Self-Contained MLP Zoo + Graph Dataset Notebook

This notebook is fully standalone:
- trains **1000 MLPs** on the **same polynomial task**
- uses the **same activation** for all models
- varies hidden layers/widths across models
- converts trained checkpoints into weighted graph files for GNN-VAE work
- shows progress bars while running

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
    else:
        print(f'{import_name} already installed.')

ensure_package('torch', 'torch')
ensure_package('tqdm', 'tqdm')

In [ ]:
from __future__ import annotations

import csv
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

print('Torch version:', torch.__version__)

In [ ]:
# -------------------------
# User configuration
# -------------------------
CONFIG = {
    'output_dir': 'model_zoo_poly_1000',
    'num_models': 1000,
    'degree': 5,
    'coeff_scale': 1.5,
    'x_min': -2.0,
    'x_max': 2.0,
    'train_size': 2048,
    'val_size': 512,
    'test_size': 1024,
    'train_noise_std': 0.03,
    'hidden_archs': ['32-32', '64-64', '128-64', '128-128', '256-128-64', '256-256-128-64'],
    'activation': 'relu',
    'optimizers': ['adam', 'adamw'],
    'batch_sizes': [64, 128, 256],
    'lr_min': 1e-4,
    'lr_max': 3e-3,
    'weight_decay_min': 1e-8,
    'weight_decay_max': 1e-3,
    'max_epochs': 500,
    'patience': 60,
    'seed': 7,
    'print_every': 20,
}

GRAPH_CONFIG = {
    'output_subdir': 'graph_zoo',
    'bidirectional': True,
    'include_layer_position_in_edge_attr': True,
    'require_fixed_architecture': False,
    'val_ratio': 0.1,
    'test_ratio': 0.1,
    'split_seed': 123,
}

CONFIG, GRAPH_CONFIG

In [ ]:
# -------------------------
# Training helpers
# -------------------------
ACTIVATIONS = {
    'relu': nn.ReLU,
    'tanh': nn.Tanh,
    'gelu': nn.GELU,
    'silu': nn.SiLU,
    'elu': nn.ELU,
}

def parse_hidden_arch(raw: str) -> list[int]:
    tokens = raw.replace('x', '-').replace(',', '-').split('-')
    dims = [int(t) for t in tokens if t.strip()]
    if not dims:
        raise ValueError(f'Invalid hidden architecture: {raw!r}')
    return dims

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def choose_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def sample_log_uniform(rng: random.Random, low: float, high: float) -> float:
    if low <= 0 or high <= 0 or high < low:
        raise ValueError('Log-uniform bounds must satisfy 0 < low <= high.')
    return math.exp(rng.uniform(math.log(low), math.log(high)))

def sample_coefficients(degree: int, scale: float, generator: torch.Generator) -> torch.Tensor:
    coeffs = torch.empty(degree + 1).uniform_(-scale, scale, generator=generator)
    if degree >= 1 and torch.max(torch.abs(coeffs[1:])).item() < 1e-6:
        coeffs[1] = scale
    return coeffs

def polynomial_forward(x: torch.Tensor, coeffs: torch.Tensor) -> torch.Tensor:
    y = torch.zeros_like(x)
    for c in reversed(coeffs):
        y = y * x + c
    return y

def sample_xy(n: int, coeffs: torch.Tensor, x_min: float, x_max: float, noise_std: float, generator: torch.Generator):
    x = (x_max - x_min) * torch.rand((n, 1), generator=generator) + x_min
    y = polynomial_forward(x, coeffs)
    if noise_std > 0:
        y = y + noise_std * torch.randn((n, 1), generator=generator)
    return x.float(), y.float()

def make_mlp(input_dim: int, hidden_dims: list[int], output_dim: int, activation: str) -> nn.Module:
    if activation not in ACTIVATIONS:
        raise ValueError(f'Unknown activation: {activation}')
    layers: list[nn.Module] = []
    d = input_dim
    for h in hidden_dims:
        layers.append(nn.Linear(d, h))
        layers.append(ACTIVATIONS[activation]())
        d = h
    layers.append(nn.Linear(d, output_dim))
    return nn.Sequential(*layers)

@torch.no_grad()
def predict(model: nn.Module, x: torch.Tensor, batch_size: int, device: torch.device) -> torch.Tensor:
    model.eval()
    out = []
    for i in range(0, x.size(0), batch_size):
        xb = x[i:i+batch_size].to(device)
        out.append(model(xb).cpu())
    return torch.cat(out, dim=0)

def mse(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    return torch.mean((y_true - y_pred) ** 2).item()

def r2_score(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    ss_res = torch.sum((y_true - y_pred) ** 2)
    ss_tot = torch.sum((y_true - torch.mean(y_true)) ** 2)
    if ss_tot.item() == 0:
        return 0.0
    return (1.0 - (ss_res / ss_tot)).item()

def train_one(model: nn.Module, train_loader: DataLoader, val_x: torch.Tensor, val_y: torch.Tensor,
              optimizer_name: str, lr: float, weight_decay: float, max_epochs: int, patience: int,
              device: torch.device) -> dict[str, Any]:
    criterion = nn.MSELoss()
    if optimizer_name == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'adamw':
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=0.9)
    else:
        raise ValueError(f'Unknown optimizer: {optimizer_name}')

    model.to(device)
    best_val = float('inf')
    best_epoch = 0
    best_state: dict[str, torch.Tensor] = {}
    wait = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        val_pred = predict(model, val_x, batch_size=1024, device=device)
        val_mse = mse(val_y, val_pred)
        if val_mse < best_val - 1e-12:
            best_val = val_mse
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
    return {'best_val_mse': best_val, 'best_epoch': best_epoch}

def num_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


In [ ]:
def generate_model_zoo(cfg: dict[str, Any]) -> Path:
    out_dir = Path(cfg['output_dir'])
    models_dir = out_dir / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)

    set_global_seed(cfg['seed'])
    device = choose_device()
    print('Using device:', device)

    coeff_gen = torch.Generator().manual_seed(cfg['seed'] + 101)
    coeffs = sample_coefficients(cfg['degree'], cfg['coeff_scale'], coeff_gen)

    data_gen = torch.Generator().manual_seed(cfg['seed'] + 202)
    train_x, train_y = sample_xy(cfg['train_size'], coeffs, cfg['x_min'], cfg['x_max'], cfg['train_noise_std'], data_gen)
    val_x, val_y = sample_xy(cfg['val_size'], coeffs, cfg['x_min'], cfg['x_max'], 0.0, data_gen)
    test_x, test_y = sample_xy(cfg['test_size'], coeffs, cfg['x_min'], cfg['x_max'], 0.0, data_gen)

    index_rows: list[dict[str, Any]] = []
    pbar = tqdm(range(cfg['num_models']), desc='Training MLP zoo', unit='model')

    for model_id in pbar:
        model_seed = cfg['seed'] + 10_000 + model_id
        rng = random.Random(model_seed)

        hidden_dims = parse_hidden_arch(rng.choice(cfg['hidden_archs']))
        activation = cfg['activation']
        optimizer_name = rng.choice(cfg['optimizers'])
        batch_size = rng.choice(cfg['batch_sizes'])
        lr = sample_log_uniform(rng, cfg['lr_min'], cfg['lr_max'])
        wd = sample_log_uniform(rng, cfg['weight_decay_min'], cfg['weight_decay_max'])

        model = make_mlp(1, hidden_dims, 1, activation)
        train_loader = DataLoader(
            TensorDataset(train_x, train_y),
            batch_size=batch_size,
            shuffle=True,
            generator=torch.Generator().manual_seed(model_seed + 5),
        )

        train_info = train_one(
            model=model,
            train_loader=train_loader,
            val_x=val_x,
            val_y=val_y,
            optimizer_name=optimizer_name,
            lr=lr,
            weight_decay=wd,
            max_epochs=cfg['max_epochs'],
            patience=cfg['patience'],
            device=device,
        )

        train_pred = predict(model, train_x, batch_size=1024, device=device)
        val_pred = predict(model, val_x, batch_size=1024, device=device)
        test_pred = predict(model, test_x, batch_size=1024, device=device)

        train_mse = mse(train_y, train_pred)
        val_mse = mse(val_y, val_pred)
        test_mse = mse(test_y, test_pred)
        test_r2 = r2_score(test_y, test_pred)

        checkpoint_name = f'model_{model_id:05d}.pt'
        checkpoint_path = models_dir / checkpoint_name
        checkpoint = {
            'model_id': model_id,
            'seed': model_seed,
            'state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
            'architecture': {
                'input_dim': 1,
                'hidden_dims': hidden_dims,
                'output_dim': 1,
                'activation': activation,
            },
            'hyperparameters': {
                'optimizer': optimizer_name,
                'batch_size': batch_size,
                'lr': lr,
                'weight_decay': wd,
                'max_epochs': cfg['max_epochs'],
                'patience': cfg['patience'],
            },
            'metrics': {
                'best_epoch': train_info['best_epoch'],
                'train_mse': train_mse,
                'val_mse': val_mse,
                'test_mse': test_mse,
                'test_r2': test_r2,
            },
            'task': {
                'degree': cfg['degree'],
                'coefficients': [float(c) for c in coeffs.tolist()],
                'x_min': cfg['x_min'],
                'x_max': cfg['x_max'],
                'train_noise_std': cfg['train_noise_std'],
            },
        }
        torch.save(checkpoint, checkpoint_path)

        row = {
            'model_id': model_id,
            'checkpoint': str(checkpoint_path.relative_to(out_dir)),
            'seed': model_seed,
            'hidden_dims': '-'.join(str(x) for x in hidden_dims),
            'activation': activation,
            'optimizer': optimizer_name,
            'batch_size': batch_size,
            'lr': lr,
            'weight_decay': wd,
            'best_epoch': train_info['best_epoch'],
            'num_parameters': num_parameters(model),
            'train_mse': train_mse,
            'val_mse': val_mse,
            'test_mse': test_mse,
            'test_r2': test_r2,
        }
        index_rows.append(row)

        if (model_id + 1) % max(1, cfg['print_every']) == 0:
            pbar.set_postfix({
                'model': model_id + 1,
                'val_mse': f'{val_mse:.5f}',
                'test_r2': f'{test_r2:.4f}',
            })

    jsonl_path = out_dir / 'zoo_index.jsonl'
    with jsonl_path.open('w', encoding='utf-8') as f:
        for row in index_rows:
            f.write(json.dumps(row) + '\n')

    csv_path = out_dir / 'zoo_index.csv'
    with csv_path.open('w', newline='', encoding='utf-8') as f:
        fieldnames = list(index_rows[0].keys()) if index_rows else []
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(index_rows)

    task_config = {
        'seed': cfg['seed'],
        'degree': cfg['degree'],
        'coeff_scale': cfg['coeff_scale'],
        'coefficients': [float(c) for c in coeffs.tolist()],
        'x_min': cfg['x_min'],
        'x_max': cfg['x_max'],
        'train_size': cfg['train_size'],
        'val_size': cfg['val_size'],
        'test_size': cfg['test_size'],
        'train_noise_std': cfg['train_noise_std'],
    }
    with (out_dir / 'task_config.json').open('w', encoding='utf-8') as f:
        json.dump(task_config, f, indent=2)

    mean_test_mse = sum(float(r['test_mse']) for r in index_rows) / len(index_rows)
    mean_test_r2 = sum(float(r['test_r2']) for r in index_rows) / len(index_rows)
    print(f'Completed {len(index_rows)} models -> {models_dir}')
    print(f'Average test MSE: {mean_test_mse:.6f}')
    print(f'Average test R^2: {mean_test_r2:.6f}')

    return out_dir

zoo_dir = generate_model_zoo(CONFIG)
zoo_dir

In [ ]:
# Quick sanity check
models_dir = Path(CONFIG['output_dir']) / 'models'
jsonl_path = Path(CONFIG['output_dir']) / 'zoo_index.jsonl'

num_ckpts = len(list(models_dir.glob('*.pt')))
print('Checkpoints:', num_ckpts)
print('Index exists:', jsonl_path.exists())

with jsonl_path.open('r', encoding='utf-8') as f:
    first = json.loads(next(f))
print({k: first[k] for k in ['model_id', 'hidden_dims', 'activation', 'test_mse', 'test_r2']})

In [ ]:
# -------------------------
# Graph conversion helpers
# -------------------------
@dataclass(frozen=True)
class GraphEncodingConfig:
    bidirectional: bool = True
    include_layer_position_in_edge_attr: bool = True

def _extract_linear_layers(state_dict: dict[str, torch.Tensor]):
    weights, biases = [], []
    for key, tensor in state_dict.items():
        if key.endswith('.weight') and tensor.ndim == 2:
            prefix = key[:-len('.weight')]
            bkey = f'{prefix}.bias'
            if bkey in state_dict and state_dict[bkey].ndim == 1 and state_dict[bkey].shape[0] == tensor.shape[0]:
                weights.append(tensor.detach().cpu().float())
                biases.append(state_dict[bkey].detach().cpu().float())
    if not weights:
        raise ValueError('No linear layers found in state_dict.')
    return weights, biases

def _derive_layer_dims(weights: list[torch.Tensor]) -> list[int]:
    dims = [int(weights[0].shape[1])]
    dims.extend(int(w.shape[0]) for w in weights)
    return dims

def architecture_signature(layer_dims: list[int], activation: str) -> str:
    return f"dims={'-'.join(str(d) for d in layer_dims)}|act={activation}"

def _node_features(layer_dims: list[int], biases: list[torch.Tensor]) -> torch.Tensor:
    num_layers = len(layer_dims)
    max_dim = float(max(layer_dims))
    feats = []
    for layer_idx, dim in enumerate(layer_dims):
        is_input = 1.0 if layer_idx == 0 else 0.0
        is_output = 1.0 if layer_idx == num_layers - 1 else 0.0
        is_hidden = 1.0 if (not is_input and not is_output) else 0.0
        fan_in = float(layer_dims[layer_idx - 1]) if layer_idx > 0 else 0.0
        fan_out = float(layer_dims[layer_idx + 1]) if layer_idx < num_layers - 1 else 0.0
        layer_bias = biases[layer_idx - 1] if layer_idx > 0 else None

        for neuron_idx in range(dim):
            layer_pos = 0.0 if num_layers == 1 else layer_idx / float(num_layers - 1)
            neuron_pos = 0.0 if dim == 1 else neuron_idx / float(dim - 1)
            bias_val = 0.0 if layer_bias is None else float(layer_bias[neuron_idx].item())
            feats.append([
                layer_pos, neuron_pos, bias_val,
                fan_in / max_dim, fan_out / max_dim,
                is_input, is_hidden, is_output
            ])

    return torch.tensor(feats, dtype=torch.float32)

def _edge_features_and_index(layer_dims: list[int], weights: list[torch.Tensor], cfg: GraphEncodingConfig):
    offsets = []
    running = 0
    for d in layer_dims:
        offsets.append(running)
        running += d

    edge_src, edge_dst, edge_attr = [], [], []
    num_linear = len(weights)
    for linear_idx, w in enumerate(weights):
        in_dim, out_dim = int(w.shape[1]), int(w.shape[0])
        src_offset, dst_offset = offsets[linear_idx], offsets[linear_idx + 1]
        layer_pos = 0.0 if num_linear == 1 else linear_idx / float(num_linear - 1)

        for out_j in range(out_dim):
            for in_i in range(in_dim):
                val = float(w[out_j, in_i].item())
                attr = [val, abs(val), 1.0]
                if cfg.include_layer_position_in_edge_attr:
                    attr.append(layer_pos)

                src = src_offset + in_i
                dst = dst_offset + out_j
                edge_src.append(src)
                edge_dst.append(dst)
                edge_attr.append(attr)

                if cfg.bidirectional:
                    rev_attr = [val, abs(val), -1.0]
                    if cfg.include_layer_position_in_edge_attr:
                        rev_attr.append(layer_pos)
                    edge_src.append(dst)
                    edge_dst.append(src)
                    edge_attr.append(rev_attr)

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr_tensor = torch.tensor(edge_attr, dtype=torch.float32)
    return edge_index, edge_attr_tensor

def checkpoint_to_graph(checkpoint: dict[str, Any], checkpoint_path: Path, cfg: GraphEncodingConfig) -> dict[str, Any]:
    weights, biases = _extract_linear_layers(checkpoint['state_dict'])
    layer_dims = _derive_layer_dims(weights)

    x = _node_features(layer_dims, biases)
    edge_index, edge_attr = _edge_features_and_index(layer_dims, weights, cfg)

    param_parts, param_shapes = [], []
    for w, b in zip(weights, biases):
        param_parts.append(w.reshape(-1))
        param_parts.append(b.reshape(-1))
        param_shapes.append(list(w.shape))
        param_shapes.append(list(b.shape))

    arch = checkpoint.get('architecture', {})
    activation = str(arch.get('activation', 'unknown'))

    return {
        'model_id': checkpoint.get('model_id'),
        'checkpoint_path': str(checkpoint_path),
        'x': x,
        'edge_index': edge_index,
        'edge_attr': edge_attr,
        'num_nodes': int(x.shape[0]),
        'num_edges': int(edge_index.shape[1]),
        'architecture': {
            'layer_dims': [int(v) for v in layer_dims],
            'activation': activation,
            'signature': architecture_signature(layer_dims, activation),
        },
        'metrics': checkpoint.get('metrics', {}),
        'task': checkpoint.get('task', {}),
        'param_vector': torch.cat(param_parts, dim=0).float(),
        'param_shapes': param_shapes,
    }

def build_splits(num_graphs: int, val_ratio: float, test_ratio: float, seed: int):
    if val_ratio + test_ratio >= 1.0:
        raise ValueError('val_ratio + test_ratio must be < 1.')
    ids = list(range(num_graphs))
    rng = random.Random(seed)
    rng.shuffle(ids)
    n_val = int(num_graphs * val_ratio)
    n_test = int(num_graphs * test_ratio)
    n_train = num_graphs - n_val - n_test
    return {
        'train': ids[:n_train],
        'val': ids[n_train:n_train+n_val],
        'test': ids[n_train+n_val:],
    }

In [ ]:
def build_graph_zoo(zoo_dir: Path, cfg: dict[str, Any]) -> Path:
    ckpt_paths = sorted((zoo_dir / 'models').glob('*.pt'))
    if not ckpt_paths:
        raise FileNotFoundError(f'No checkpoints found in {zoo_dir / "models"}')

    out_dir = zoo_dir / cfg['output_subdir']
    graphs_dir = out_dir / 'graphs'
    graphs_dir.mkdir(parents=True, exist_ok=True)

    enc_cfg = GraphEncodingConfig(
        bidirectional=cfg['bidirectional'],
        include_layer_position_in_edge_attr=cfg['include_layer_position_in_edge_attr'],
    )

    index_rows = []
    signatures = set()

    pbar = tqdm(enumerate(ckpt_paths), total=len(ckpt_paths), desc='Converting to graphs', unit='graph')
    for i, ckpt_path in pbar:
        checkpoint = torch.load(ckpt_path, map_location='cpu')
        graph = checkpoint_to_graph(checkpoint, ckpt_path.resolve(), enc_cfg)
        signatures.add(graph['architecture']['signature'])

        graph_file = graphs_dir / f'graph_{i:05d}.pt'
        torch.save(graph, graph_file)

        m = graph.get('metrics', {})
        row = {
            'graph_id': i,
            'model_id': graph.get('model_id'),
            'graph_file': str(graph_file.relative_to(out_dir)),
            'checkpoint_file': str(ckpt_path.relative_to(zoo_dir)),
            'num_nodes': graph['num_nodes'],
            'num_edges': graph['num_edges'],
            'architecture_signature': graph['architecture']['signature'],
            'val_mse': m.get('val_mse'),
            'test_mse': m.get('test_mse'),
            'test_r2': m.get('test_r2'),
        }
        index_rows.append(row)

        if (i + 1) % 25 == 0:
            pbar.set_postfix({'nodes': row['num_nodes'], 'edges': row['num_edges']})

    if cfg['require_fixed_architecture'] and len(signatures) > 1:
        raise ValueError('Multiple architectures found with require_fixed_architecture=True')

    index_path = out_dir / 'graph_index.jsonl'
    with index_path.open('w', encoding='utf-8') as f:
        for row in index_rows:
            f.write(json.dumps(row) + '\n')

    splits = build_splits(len(index_rows), cfg['val_ratio'], cfg['test_ratio'], cfg['split_seed'])
    with (out_dir / 'splits.json').open('w', encoding='utf-8') as f:
        json.dump(splits, f, indent=2)

    summary = {
        'num_graphs': len(index_rows),
        'num_unique_architectures': len(signatures),
        'architectures': sorted(signatures),
        'edge_attr_dim': 4 if cfg['include_layer_position_in_edge_attr'] else 3,
        'node_attr_dim': 8,
        'bidirectional': cfg['bidirectional'],
    }
    with (out_dir / 'summary.json').open('w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    print(f'Graph dataset written to: {out_dir}')
    return out_dir

graph_dir = build_graph_zoo(Path(CONFIG['output_dir']), GRAPH_CONFIG)
graph_dir

In [ ]:
# Preview one graph artifact
example_graph_path = Path(CONFIG['output_dir']) / GRAPH_CONFIG['output_subdir'] / 'graphs' / 'graph_00000.pt'
g = torch.load(example_graph_path, map_location='cpu')
print('Graph file:', example_graph_path)
print('x shape:', tuple(g['x'].shape))
print('edge_index shape:', tuple(g['edge_index'].shape))
print('edge_attr shape:', tuple(g['edge_attr'].shape))
print('architecture:', g['architecture'])